# Daily Challenge: Build a Retrieval Augmented Generation (RAG) System
## Week 8 — Day 3 | DI GenAI & Machine Learning Bootcamp 2026

**What you'll build:**  
A functional RAG system that answers questions using the `databricks/databricks-dolly-15k` knowledge base.

**Stack:**
- **LangChain** — orchestration (loader, splitter, vector store, QA chain)
- **`sentence-transformers/all-MiniLM-l6-v2`** — text embeddings
- **FAISS** — vector store and similarity search
- **`Intel/dynamic_tinybert`** — extractive question-answering LLM

```
Dolly-15k dataset
      ↓  HuggingFaceDatasetLoader
Documents (page_content = context)
      ↓  RecursiveCharacterTextSplitter (size=1000, overlap=150)
Chunks
      ↓  all-MiniLM-l6-v2 embeddings → FAISS index
Vector store
      ↓  retriever (k=4)
Relevant context
      ↓  Intel/dynamic_tinybert (QA pipeline)
Answer
```

## Step 1 — Set Up the Environment

In [ ]:
# Install all required libraries
!pip install -q langchain
!pip install -q torch
!pip install -q transformers
!pip install -q sentence-transformers
!pip install -Uq datasets
!pip install -q faiss-cpu
!pip install -U langchain-community langchain-huggingface
print("All libraries installed ✓")

### Hugging Face Transformers — Quick Primer

The `transformers` library makes it trivial to load pre-trained models like BERT or GPT without training from scratch:

In [ ]:
# Quick HuggingFace primer (as shown in the course notes)
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model     = AutoModel.from_pretrained("bert-base-uncased")

inputs  = tokenizer("Hello Hugging Face!", return_tensors="pt")
outputs = model(**inputs)

print(f"Input tokens     : {tokenizer.convert_ids_to_tokens(inputs['input_ids'][0].tolist())}")
print(f"last_hidden_state: shape={outputs.last_hidden_state.shape}  "
      f"(batch=1, seq_len={outputs.last_hidden_state.shape[1]}, hidden=768)")
print("\n→ Each token is now a 768-dim vector representing its contextual meaning.")

## Step 2 — Load the Dataset

`databricks/databricks-dolly-15k` is a high-quality instruction-following dataset with 15 000 records.  
The `context` column contains the passage of text the answer should come from.

In [ ]:
from langchain_community.document_loaders import HuggingFaceDatasetLoader

dataset_name        = "databricks/databricks-dolly-15k"
page_content_column = "context"

print(f"Loading dataset: {dataset_name}")
print(f"Content column : {page_content_column}")
print("(This may take ~1 min on first run — model weights are downloaded)")

loader = HuggingFaceDatasetLoader(dataset_name, page_content_column)
data   = loader.load()

print(f"\nDocuments loaded : {len(data):,}")
print("\n--- First 2 documents ---")
print(data[:2])

In [ ]:
# Inspect document structure
print("=== Document 0 ===")
print(f"page_content : {data[0].page_content[:400]}")
print(f"metadata     : {data[0].metadata}")

# Count non-empty documents (context column can be empty for some rows)
non_empty = [d for d in data if d.page_content and d.page_content.strip()]
print(f"\nTotal documents    : {len(data):,}")
print(f"Non-empty context  : {len(non_empty):,}")
print(f"Empty context      : {len(data) - len(non_empty):,}  (these will produce no retrievable chunks)")

## Step 3 — Split the Documents

Language models have a token limit. Splitting into overlapping chunks (overlap=150) prevents losing context at boundaries.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size    = 1000,
    chunk_overlap = 150,
)

# Only split non-empty documents to avoid creating empty chunks
docs = text_splitter.split_documents(non_empty)

print(f"Non-empty documents  : {len(non_empty):,}")
print(f"Chunks after split   : {len(docs):,}")
print(f"Avg chunk length     : {sum(len(d.page_content) for d in docs) // len(docs)} chars")
print()
print("--- First document chunk ---")
print(docs[0])

## Step 4 — Embed the Text

`sentence-transformers/all-MiniLM-l6-v2` converts text into 384-dim vectors that capture semantic meaning.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

modelPath      = "sentence-transformers/all-MiniLM-l6-v2"
model_kwargs   = {"device": "cpu"}
encode_kwargs  = {"normalize_embeddings": False}

print(f"Loading embedding model: {modelPath}")
embeddings = HuggingFaceEmbeddings(
    model_name    = modelPath,
    model_kwargs  = model_kwargs,
    encode_kwargs = encode_kwargs,
)
print("Embedding model loaded ✓")

# Test embedding creation
text         = "This is a test document."
query_result = embeddings.embed_query(text)
print(f"\nTest embedding:")
print(f"  Input     : '{text}'")
print(f"  Dimension : {len(query_result)}")
print(f"  First 3   : {query_result[:3]}")

## Step 5 — Create the FAISS Vector Store

FAISS indexes all chunk embeddings for fast approximate nearest-neighbour search at query time.

In [ ]:
from langchain_community.vectorstores import FAISS

# We index a representative subset (first 3000 chunks) to keep build time reasonable
# On a full GPU machine you can remove the slice
index_docs = docs[:3000]

print(f"Building FAISS index from {len(index_docs):,} chunks…")
print("(This may take 2–5 minutes on CPU)")

db = FAISS.from_documents(index_docs, embeddings)

print(f"\nFAISS vector store built ✓")
print(f"  Vectors indexed : {db.index.ntotal:,}")

## Step 6 — Prepare the LLM: `Intel/dynamic_tinybert`

`Intel/dynamic_tinybert` is a compact BERT-based model fine-tuned for extractive question answering.  
It takes a `(question, context)` pair and extracts the answer span from the context.

In [ ]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline
from langchain_huggingface import HuggingFacePipeline

model_name = "Intel/dynamic_tinybert"

print(f"Loading tokenizer for '{model_name}'…")
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    padding    = True,
    truncation = True,
    max_length = 512,
)

print(f"Loading QA model '{model_name}'…")
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded ✓  ({n_params/1e6:.1f}M parameters)")

In [ ]:
# Create the HuggingFace question-answering pipeline
Youtubeer = pipeline(
    "question-answering",
    model       = model_name,
    tokenizer   = tokenizer,
    return_tensors = "pt",
)

# Wrap in LangChain so it can be used inside a chain
llm = HuggingFacePipeline(
    pipeline     = Youtubeer,
    model_kwargs = {"temperature": 0.7, "max_length": 512},
)

print("LangChain HuggingFacePipeline wrapper created ✓")
print(f"  Model    : {model_name}")
print(f"  Task     : question-answering (extractive)")

In [ ]:
# Quick standalone test of the QA pipeline before wiring into the chain
test_result = Youtubeer(
    question = "What is BERT?",
    context  = (
        "BERT (Bidirectional Encoder Representations from Transformers) is a "
        "pre-trained natural language processing model developed by Google. "
        "It is designed to understand the context of words in a sentence by "
        "reading the text both left-to-right and right-to-left."
    ),
)
print("=== Standalone QA test ===")
print(f"Question : What is BERT?")
print(f"Answer   : {test_result['answer']}")
print(f"Score    : {test_result['score']:.4f}")

## Step 7 — Build the Retrieval QA Chain

The `RetrievalQA` chain ties everything together:
1. Retriever fetches the top-k most relevant chunks from FAISS.
2. `chain_type="refine"` iteratively feeds each chunk to the LLM, refining the answer at each step.

In [ ]:
from langchain.chains import RetrievalQA

# Build the retriever — returns top 4 semantically similar chunks
retriever = db.as_retriever(search_kwargs={"k": 4})

# Build the RetrievalQA chain
qa = RetrievalQA.from_chain_type(
    llm                  = llm,
    chain_type           = "refine",
    retriever            = retriever,
    return_source_documents = False,
)

print("RetrievalQA chain built ✓")
print(f"  chain_type : refine  (iteratively refines answer over each retrieved chunk)")
print(f"  retriever  : FAISS, k=4")
print(f"  LLM        : Intel/dynamic_tinybert (extractive QA)")

## Step 8 — Test the RAG System

In [ ]:
# Helper: run query and show retrieved chunks
def ask(question: str) -> None:
    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print("=" * 60)

    # Show retrieved chunks first
    chunks_retrieved = retriever.invoke(question)
    print(f"Retrieved {len(chunks_retrieved)} chunks:")
    for i, c in enumerate(chunks_retrieved, 1):
        print(f"  [{i}] {c.page_content[:120]}…")

    # Run the RAG chain
    try:
        result = qa.invoke({"query": question})
        answer = result["result"] if isinstance(result, dict) else str(result)
    except Exception as e:
        # Fallback: run the QA pipeline directly on the best retrieved chunk
        best_context = chunks_retrieved[0].page_content if chunks_retrieved else ""
        raw          = Youtubeer(question=question, context=best_context)
        answer       = raw["answer"]

    print(f"\nAnswer: {answer}")
    print()


print("ask() helper defined ✓")

In [ ]:
# Required test query from the challenge
question = "What is cheesemaking?"
result   = qa.invoke({"query": question})
print(f"Question : {question}")
print(f"Result   : {result}")

# Also print just the answer field if result is a dict
if isinstance(result, dict) and "result" in result:
    print(f"\nAnswer   : {result['result']}")

In [ ]:
# Additional test queries — exploring different Dolly topics
ask("What is the capital of France?")

In [ ]:
ask("How does photosynthesis work?")

In [ ]:
ask("Who invented the telephone?")

In [ ]:
ask("What are the main ingredients in bread?")

## Bonus — Direct Pipeline Queries on Retrieved Context

For maximum transparency, we can also directly use the QA pipeline on the best retrieved chunk and see the confidence score.

In [ ]:
def ask_direct(question: str, top_k: int = 4) -> None:
    """Run QA pipeline directly on each retrieved chunk and show scores."""
    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print("=" * 60)

    chunks_retrieved = retriever.invoke(question)
    answers          = []

    for chunk in chunks_retrieved:
        if not chunk.page_content.strip():
            continue
        try:
            res = Youtubeer(question=question, context=chunk.page_content)
            answers.append((res["score"], res["answer"], chunk.page_content[:100]))
        except Exception:
            pass

    # Sort by confidence score
    answers.sort(key=lambda x: x[0], reverse=True)

    print(f"{'Rank':<5} {'Score':<10} {'Answer':<40} Context snippet")
    print("-" * 90)
    for i, (score, answer, ctx) in enumerate(answers, 1):
        print(f"{i:<5} {score:<10.4f} {answer[:38]:<40} {ctx}…")

    if answers:
        best = answers[0]
        print(f"\nBest answer (score={best[0]:.4f}): {best[1]}")


ask_direct("What is cheesemaking?")
ask_direct("What causes earthquakes?")

## Summary & Key Takeaways

| Step | Component | Detail |
|---|---|---|
| 1 | Install libs | `langchain`, `faiss-cpu`, `sentence-transformers`, `transformers`, `datasets` |
| 2 | Load | `HuggingFaceDatasetLoader("databricks/databricks-dolly-15k", "context")` |
| 3 | Split | `RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)` |
| 4 | Embed | `HuggingFaceEmbeddings("sentence-transformers/all-MiniLM-l6-v2")` → 384-dim |
| 5 | Index | `FAISS.from_documents(docs, embeddings)` |
| 6 | LLM | `Intel/dynamic_tinybert` — extractive QA via `AutoModelForQuestionAnswering` |
| 7 | Chain | `RetrievalQA.from_chain_type(llm, chain_type="refine", retriever)` |
| 8 | Query | `qa.invoke({"query": question})` |

### RAG vs Plain LLM

| Aspect | Plain LLM | RAG |
|---|---|---|
| Knowledge source | Training data (static) | External vector store (updatable) |
| Hallucination risk | High | Lower (grounded in retrieved docs) |
| Freshness | Frozen at cutoff | Can add new docs at any time |
| Transparency | Black box | Sources visible — auditable |

### `chain_type` options in LangChain

| Type | Behaviour |
|---|---|
| `stuff` | All chunks concatenated into one prompt — simplest, fast |
| `refine` | Iterates over chunks, refining the answer at each step — better for QA |
| `map_reduce` | Summarizes each chunk independently, then combines — best for long docs |
| `map_rerank` | Scores each chunk independently, picks the highest — fast but may miss context |